### 028

In [125]:
# 965명의 유저가 있다.
# 01, 02 equipment 존재
# 01에는 F, L15, L30, R15, R30
# 02에는 F, L, R
print(965 * (5 + 3))
# F에선 2장씩, 나머지에선 1장씩
print(965 * (2 + 2 + 1 + 1 + 1 + 1 + 1 + 1))

7720
9650


In [138]:
pig_root = '/home/work/hocheol_dir/workspace/datasets/v0.2.2_data/pigment'

In [139]:
import os
import pandas as pd
# meta data 가져오기

label_path = "/home/work/hocheol_dir/workspace/datasets/v0.2_data/pigment/028_data/user_info_converted.csv"
label_df = pd.read_csv(label_path)
label_df['id'] = label_df['id'].apply(lambda x:str(x).zfill(4))
label_df = label_df[['id', 'l_cheek_pigmentation', 'r_cheek_pigmentation']]
label_df.rename(columns={'l_cheek_pigmentation': 'left_label', 'r_cheek_pigmentation': 'right_label'}, inplace=True)
label_df.head()

,id,left_label,right_label
0,0001,3,3
1,0002,3,3
2,0003,1,0
3,0004,3,3
4,0006,3,3


In [140]:
len(os.listdir(os.path.join(pig_root, '028_data/data')))

9650

In [160]:
# image path 가져오기
import os
import re

id_set = set()
exts = set()

for image in os.listdir(os.path.join(pig_root, '028_data/data')):
    exts.add(image.split('.')[-1])
    user_id = re.search(r'[0-9]{4}_[0-9]{2}_[A-Z][0-9]*.*', image).group(0)
    equipment = image.split('_')[2]
    id_set.add(user_id)
    
len(id_set), list(id_set)[:2], exts

(7720, ['0869_02_L.png', '0023_01_L30.png'], {'png'})

In [174]:
F01_df = pd.DataFrame(columns=['user_eq', 'left_path', 'right_path'])
F02_df = pd.DataFrame(columns=['user_eq', 'left_path', 'right_path'])
L15_df = pd.DataFrame(columns=['user_eq', 'left_path'])
L30_df = pd.DataFrame(columns=['user_eq', 'left_path'])
R15_df = pd.DataFrame(columns=['user_eq', 'right_path'])
R30_df = pd.DataFrame(columns=['user_eq', 'right_path'])
L_df = pd.DataFrame(columns=['user_eq', 'left_path'])
R_df = pd.DataFrame(columns=['user_eq', 'right_path'])

for user_id in id_set:
    if user_id.endswith('01_F.png'):
        F01_df.loc[len(F01_df)] = (user_id[:7], f"028_data/data/left_{user_id}", f"028_data/data/right_{user_id}")
    if user_id.endswith('02_F.png'):
        F02_df.loc[len(F02_df)] = (user_id[:7], f"028_data/data/left_{user_id}", f"028_data/data/right_{user_id}")
    if user_id.endswith('_L15.png'):
        L15_df.loc[len(L15_df)] = (user_id[:7], f"028_data/data/left_{user_id}")
    if user_id.endswith('_L30.png'):
        L30_df.loc[len(L30_df)] = (user_id[:7], f"028_data/data/left_{user_id}")
    if user_id.endswith('_R15.png'):
        R15_df.loc[len(R15_df)] = (user_id[:7], f"028_data/data/right_{user_id}")
    if user_id.endswith('_R30.png'):
        R30_df.loc[len(R30_df)] = (user_id[:7], f"028_data/data/right_{user_id}")
    if user_id.endswith('_L.png'):
        L_df.loc[len(L_df)] = (user_id[:7], f"028_data/data/left_{user_id}")
    if user_id.endswith('_R.png'):
        R_df.loc[len(R_df)] = (user_id[:7], f"028_data/data/right_{user_id}")
        
len(F01_df), len(F02_df), len(L15_df), len(L30_df), len(R15_df), len(R30_df), len(L_df), len(R_df)

(965, 965, 965, 965, 965, 965, 965, 965)

In [175]:
# 좌우 15도, 30도 틀어진거 합체
merged_15 = pd.merge(L15_df, R15_df, on='user_eq', how='outer')
merged_30 = pd.merge(L30_df, R30_df, on='user_eq', how='outer')
merged_LR = pd.merge(L_df, R_df, on='user_eq', how='outer')
len(merged_15), len(merged_30), len(merged_LR)

(965, 965, 965)

In [176]:
merged_df = pd.concat([F01_df, F02_df, merged_15, merged_30, merged_LR], ignore_index=True)
len(merged_df)

4825

In [177]:
merged_df['user_id'] = merged_df['user_eq'].apply(lambda x:x.split('_')[0])
merged_df['equipment'] = merged_df['user_eq'].apply(lambda x:x.split('_')[1])
merged_df.sample(3)

,user_eq,left_path,right_path,user_id,equipment
1431,0569_02,028_data/data/left_0569_02_F.png,028_data/data/right_0569_02_F.png,0569,02
2620,0789_01,028_data/data/left_0789_01_L15.png,028_data/data/right_0789_01_R15.png,0789,01
4069,0243_02,028_data/data/left_0243_02_L.png,028_data/data/right_0243_02_R.png,0243,02


In [178]:
label_merged_df = pd.merge(merged_df, label_df, left_on='user_id', right_on='id')
label_merged_df.drop(columns=['id'], inplace=True)
label_merged_df = label_merged_df[['user_id', 'equipment', 'left_path', 'right_path', 'left_label', 'right_label']]
label_merged_df.sort_values(by=['user_id', 'equipment', 'left_path'], ignore_index=True, inplace=True)
label_merged_df.isnull().sum(), label_merged_df.isna().sum()

(user_id        0
 equipment      0
 left_path      0
 right_path     0
 left_label     0
 right_label    0
 dtype: int64,
 user_id        0
 equipment      0
 left_path      0
 right_path     0
 left_label     0
 right_label    0
 dtype: int64)

In [179]:
import json

json_path = os.path.join(pig_root, '028_data', 'label', 'pig_label.json')
os.makedirs(os.path.dirname(json_path), exist_ok=True)
with open(json_path, 'w') as f:
    json.dump(label_merged_df.to_dict(orient='records'), f, indent=2, ensure_ascii=False)

### 030

In [180]:
import os
import pandas as pd
# meta data 가져오기

label_path = "/home/work/hocheol_dir/workspace/datasets/v0.2_data/pigment/030_data/NIA_Korean_wholebody_pigmentation_250616.csv"
label_df = pd.read_csv(label_path)
label_df.rename(columns={'filename': 'user_id'})
label_df.drop(columns=['성별', '나이'], inplace=True)
label_df.head()

,filename,right_label,left_label
0,0001_face,0,0
1,0002_face,0,1
2,0003_face,1,0
3,0004_face,0,0
4,0005_face,1,1


In [181]:
len(os.listdir(os.path.join(pig_root, '030_data/data')))

3248

In [182]:
# image path 가져오기
import os
import re

pig_root = '/home/work/hocheol_dir/workspace/datasets/v0.2.2_data/pigment'

exts = set()

for image in os.listdir(os.path.join(pig_root, '030_data/data')):
    exts.add(image.split('.')[-1])
    
exts

{'png'}

In [183]:
id_set = set()
image_df = pd.DataFrame(columns=['user_id', 'left_path', 'right_path'])

for image in os.listdir(os.path.join(pig_root, '030_data/data')):
    image_path = os.path.join('030_data/data', image)
    user_id = image.split('_')[1]

    # user_id가 image_df에 존재하는지 확인
    if user_id in image_df.loc[:, 'user_id'].values:
        # 왼쪽 볼인지 오른쪽 볼인지 확인하여 경로 추가
        if 'left' in image:
            image_df.loc[image_df['user_id'] == user_id, 'left_path'] = image_path
        elif 'right' in image:
            image_df.loc[image_df['user_id'] == user_id, 'right_path'] = image_path
        else:
            print(f"Unknown image format: {image}")
    else:
        # 왼쪽 볼인지 오른쪽 볼인지 확인하여 새 행 추가
        if 'left' in image:
            image_df.loc[len(image_df)] = (user_id, image_path, None)
        elif 'right' in image:
            image_df.loc[len(image_df)] = (user_id, None, image_path)
        else:
            print(f"Unknown image format: {image}")

len(image_df)

1624

In [187]:
# image df 완성
image_df.sample(3)

,user_id,left_path,right_path
475,0608,030_data/data/left_0608_face.png,030_data/data/right_0608_face.png
1292,1570,030_data/data/left_1570_face.png,030_data/data/right_1570_face.png
172,0212,030_data/data/left_0212_face.png,030_data/data/right_0212_face.png


In [189]:
# image_df
label_df['user_id'] = label_df['filename'].apply(lambda x:x.split('_')[0])
label_df

,filename,right_label,left_label,user_id
0,0001_face,0,0,0001
1,0002_face,0,1,0002
2,0003_face,1,0,0003
3,0004_face,0,0,0004
4,0005_face,1,1,0005
...,...,...,...,...
1619,1976_face,2,1,1976
1620,1977_face,2,3,1977
1621,1978_face,1,2,1978
1622,1979_face,4,4,1979


In [190]:
merged_df = pd.merge(image_df, label_df, on='user_id')
len(merged_df)

1624

In [191]:
failed_paths = set()
for idx, row in merged_df.iterrows():
    if not os.path.exists(os.path.join(pig_root, row['left_path'])):
        failed_paths.add(row['left_path'])
    if not os.path.exists(os.path.join(pig_root, row['right_path'])):
        failed_paths.add(row['right_path'])

len(failed_paths)

0

In [192]:
merged_df = merged_df[['user_id', 'left_path', 'right_path', 'left_label', 'right_label']]
merged_df

,user_id,left_path,right_path,left_label,right_label
0,0082,030_data/data/left_0082_face.png,030_data/data/right_0082_face.png,0,1
1,0034,030_data/data/left_0034_face.png,030_data/data/right_0034_face.png,1,0
2,0081,030_data/data/left_0081_face.png,030_data/data/right_0081_face.png,1,1
3,0067,030_data/data/left_0067_face.png,030_data/data/right_0067_face.png,0,1
4,0013,030_data/data/left_0013_face.png,030_data/data/right_0013_face.png,0,0
...,...,...,...,...,...
1619,1979,030_data/data/left_1979_face.png,030_data/data/right_1979_face.png,4,4
1620,1975,030_data/data/left_1975_face.png,030_data/data/right_1975_face.png,2,2
1621,1978,030_data/data/left_1978_face.png,030_data/data/right_1978_face.png,2,1
1622,1977,030_data/data/left_1977_face.png,030_data/data/right_1977_face.png,3,2


In [193]:
import json

json_path = os.path.join(pig_root, '030_data', 'label', 'pig_label.json')
os.makedirs(os.path.dirname(json_path), exist_ok=True)
with open(json_path, 'w') as f:
    json.dump(merged_df.to_dict(orient='records'), f, indent=2, ensure_ascii=False)

### 034

In [195]:
import os
import pandas as pd
# meta data 가져오기

label_path = "/home/work/hocheol_dir/workspace/datasets/v0.2_data/pigment/034_data/results_250507.csv"
label_df = pd.read_csv(label_path)

label_df.head()

,filename,label
0,left_cheek_0000_CRS_19_01.jpg,0
1,left_cheek_0000_CRS_46_01.jpg,0
2,left_cheek_0001_CRS_19_01.jpg,1
3,left_cheek_0001_CRS_46_01.jpg,1
4,left_cheek_0002_CRS_19_01.jpg,0


In [196]:
label_df['user_id'] = label_df['filename'].apply(lambda x: x[x.find('cheek_')+6:-4])
left_label_df = label_df[label_df['filename'].str.contains('left_cheek')].sort_values(by='user_id', ignore_index=True)
right_label_df = label_df[label_df['filename'].str.contains('right_cheek')].sort_values(by='user_id', ignore_index=True)

In [197]:
left_label_df.rename(columns={'label': 'left_label'}, inplace=True)
right_label_df.rename(columns={'label': 'right_label'}, inplace=True)

label_df = pd.merge(left_label_df[['user_id', 'filename', 'left_label']], right_label_df[['user_id', 'right_label']], on='user_id')
label_df

,user_id,filename,left_label,right_label
0,0000_CRS_19_01,left_cheek_0000_CRS_19_01.jpg,0,0
1,0000_CRS_46_01,left_cheek_0000_CRS_46_01.jpg,0,0
2,0001_CRS_19_01,left_cheek_0001_CRS_19_01.jpg,1,0
3,0001_CRS_46_01,left_cheek_0001_CRS_46_01.jpg,1,0
4,0002_CRS_19_01,left_cheek_0002_CRS_19_01.jpg,0,1
...,...,...,...,...
5395,2697_CRS_46_01,left_cheek_2697_CRS_46_01.jpg,0,0
5396,2698_CRS_19_01,left_cheek_2698_CRS_19_01.jpg,1,1
5397,2698_CRS_46_01,left_cheek_2698_CRS_46_01.jpg,1,1
5398,2699_CRS_19_01,left_cheek_2699_CRS_19_01.jpg,2,2


In [198]:
len(os.listdir(os.path.join(pig_root, '034_data/data')))

10800

In [199]:
# image path 가져오기
import os
import re

pig_root = '/home/work/hocheol_dir/workspace/datasets/v0.2.2_data/pigment'

exts = set()

for image in os.listdir(os.path.join(pig_root, '034_data/data')):
    exts.add(image.split('.')[-1])
    
exts

{'png'}

In [200]:
left_image_df = pd.DataFrame(columns=['user_id', 'left_path'])
right_image_df = pd.DataFrame(columns=['user_id', 'right_path'])

for image in os.listdir(os.path.join(pig_root, '034_data/data')):
    image_path = os.path.join('034_data/data', image)
    if 'left' in image:
        user_id = image[5:-4]
        left_image_df.loc[len(left_image_df)] = (user_id, image_path)
    elif 'right' in image:
        user_id = image[6:-4]
        right_image_df.loc[len(right_image_df)] = (user_id, image_path)
    else:
        print(f"Unknown image format: {image}")
        continue

len(left_image_df), len(right_image_df)

(5400, 5400)

In [201]:
image_df = pd.merge(left_image_df, right_image_df, on='user_id', how='outer')

In [202]:
image_df.isna().sum(), image_df.isnull().sum()

(user_id       0
 left_path     0
 right_path    0
 dtype: int64,
 user_id       0
 left_path     0
 right_path    0
 dtype: int64)

In [203]:
# image_df
label_df.drop(columns=['filename'], inplace=True)
label_df

,user_id,left_label,right_label
0,0000_CRS_19_01,0,0
1,0000_CRS_46_01,0,0
2,0001_CRS_19_01,1,0
3,0001_CRS_46_01,1,0
4,0002_CRS_19_01,0,1
...,...,...,...
5395,2697_CRS_46_01,0,0
5396,2698_CRS_19_01,1,1
5397,2698_CRS_46_01,1,1
5398,2699_CRS_19_01,2,2


In [209]:
merged_df = pd.merge(image_df, label_df, on='user_id')
len(merged_df)

5400

In [210]:
merged_df

,user_id,left_path,right_path,left_label,right_label
0,0000_CRS_19_01,034_data/data/left_0000_CRS_19_01.png,034_data/data/right_0000_CRS_19_01.png,0,0
1,0000_CRS_46_01,034_data/data/left_0000_CRS_46_01.png,034_data/data/right_0000_CRS_46_01.png,0,0
2,0001_CRS_19_01,034_data/data/left_0001_CRS_19_01.png,034_data/data/right_0001_CRS_19_01.png,1,0
3,0001_CRS_46_01,034_data/data/left_0001_CRS_46_01.png,034_data/data/right_0001_CRS_46_01.png,1,0
4,0002_CRS_19_01,034_data/data/left_0002_CRS_19_01.png,034_data/data/right_0002_CRS_19_01.png,0,1
...,...,...,...,...,...
5395,2697_CRS_46_01,034_data/data/left_2697_CRS_46_01.png,034_data/data/right_2697_CRS_46_01.png,0,0
5396,2698_CRS_19_01,034_data/data/left_2698_CRS_19_01.png,034_data/data/right_2698_CRS_19_01.png,1,1
5397,2698_CRS_46_01,034_data/data/left_2698_CRS_46_01.png,034_data/data/right_2698_CRS_46_01.png,1,1
5398,2699_CRS_19_01,034_data/data/left_2699_CRS_19_01.png,034_data/data/right_2699_CRS_19_01.png,2,2


In [211]:
merged_df['file_id'] = merged_df['user_id']

In [212]:
merged_df['user_id'] = merged_df['user_id'].apply(lambda x: x.split('_')[0])
merged_df

,user_id,left_path,right_path,left_label,right_label,file_id
0,0000,034_data/data/left_0000_CRS_19_01.png,034_data/data/right_0000_CRS_19_01.png,0,0,0000_CRS_19_01
1,0000,034_data/data/left_0000_CRS_46_01.png,034_data/data/right_0000_CRS_46_01.png,0,0,0000_CRS_46_01
2,0001,034_data/data/left_0001_CRS_19_01.png,034_data/data/right_0001_CRS_19_01.png,1,0,0001_CRS_19_01
3,0001,034_data/data/left_0001_CRS_46_01.png,034_data/data/right_0001_CRS_46_01.png,1,0,0001_CRS_46_01
4,0002,034_data/data/left_0002_CRS_19_01.png,034_data/data/right_0002_CRS_19_01.png,0,1,0002_CRS_19_01
...,...,...,...,...,...,...
5395,2697,034_data/data/left_2697_CRS_46_01.png,034_data/data/right_2697_CRS_46_01.png,0,0,2697_CRS_46_01
5396,2698,034_data/data/left_2698_CRS_19_01.png,034_data/data/right_2698_CRS_19_01.png,1,1,2698_CRS_19_01
5397,2698,034_data/data/left_2698_CRS_46_01.png,034_data/data/right_2698_CRS_46_01.png,1,1,2698_CRS_46_01
5398,2699,034_data/data/left_2699_CRS_19_01.png,034_data/data/right_2699_CRS_19_01.png,2,2,2699_CRS_19_01


In [213]:
merged_df = merged_df[['user_id', 'file_id', 'left_path', 'right_path', 'left_label', 'right_label']]

In [214]:
failed_paths = set()
for idx, row in merged_df.iterrows():
    if not os.path.exists(os.path.join(pig_root, row['left_path'])):
        failed_paths.add(row['left_path'])
    if not os.path.exists(os.path.join(pig_root, row['right_path'])):
        failed_paths.add(row['right_path'])

len(failed_paths)

0

In [215]:
import json

json_path = os.path.join(pig_root, '034_data', 'label', 'pig_label.json')
os.makedirs(os.path.dirname(json_path), exist_ok=True)
with open(json_path, 'w') as f:
    json.dump(merged_df.to_dict(orient='records'), f, indent=2, ensure_ascii=False)

### 045

In [482]:
import os
import pandas as pd
# meta data 가져오기

label_path = "/home/work/hocheol_dir/workspace/datasets/v0.2_data/pigment/045_data/NIA_family_relation_pigmentation_250612.csv"
label_df = pd.read_csv(label_path)
label_df['filename'] = label_df['filename'].apply(lambda x: f"{os.path.splitext(x)[0]}.png")

label_df.head()

,filename,right_label,left_label
0,F0001_IND_D_18_-45_01.png,NaN,0.0
1,F0001_IND_D_18_-45_02.png,NaN,0.0
2,F0001_IND_D_18_0_01.png,0.0,0.0
3,F0001_IND_D_18_0_02.png,0.0,0.0
4,F0001_IND_D_18_45_01.png,0.0,NaN


In [483]:
len(label_df)

4622

In [484]:
import re

L_label_df = pd.DataFrame(columns=['filename', 'left_label'])
R_label_df = pd.DataFrame(columns=['filename', 'right_label'])
F_label_df = pd.DataFrame(columns=['filename', 'left_label', 'right_label'])

for i, row in label_df.iterrows():
    # print(row['filename'])
    # img_id = re.search(r'F[0-9]{4}_[A-Z]*_[A-Z]*', row['filename']).group(0)
    if '_0_' in row['filename']:
        # 정면
        F_label_df.loc[len(F_label_df)] = (row['filename'], row['left_label'], row['right_label'])
    elif '_-45_' in row['filename']:
        # left
        L_label_df.loc[len(L_label_df)] = (row['filename'], row['left_label'])
    elif '_45_' in row['filename']:
        # right
        R_label_df.loc[len(R_label_df)] = (row['filename'], row['right_label'])
    else:
        print(f"Unknown image format: {row['filename']}")

sum([len(L_label_df), len(R_label_df), len(F_label_df)]), len(label_df)

(4622, 4622)

In [485]:
len(os.listdir(os.path.join(pig_root, '045_data/data')))

6391

In [486]:
# image path 가져오기
import os
import re

pig_root = '/home/work/hocheol_dir/workspace/datasets/v0.2.2_data/pigment'

exts = set()

for image in os.listdir(os.path.join(pig_root, '045_data/data')):
    exts.add(image.split('.')[-1])
    
exts

{'png'}

In [487]:
F_image_df = pd.DataFrame(columns=['filename', 'left_path', 'right_path'])
L_image_df = pd.DataFrame(columns=['filename', 'left_path'])
R_image_df = pd.DataFrame(columns=['filename', 'right_path'])

for image in os.listdir(os.path.join(pig_root, '045_data/data')):
    image_path = os.path.join('045_data/data', image)
    filename = re.search(r'F[0-9]{4}_IND.*', image).group(0)
    direction = image.split('_')[-2]

    # print(image)
    # print(direction)
    if direction == '-45':
        L_image_df.loc[len(L_image_df)] = (filename, image_path)
    elif direction == '45':
        R_image_df.loc[len(R_image_df)] = (filename, image_path)
    elif direction == '0':
        if filename not in F_image_df['filename'].values:
            if 'left' in image:
                F_image_df.loc[len(F_image_df)] = (filename, image_path, float('nan'))
            else:
                F_image_df.loc[len(F_image_df)] = (filename, float('nan'), image_path)
        else:
            if 'left' in image:
                F_image_df.loc[F_image_df['filename'] == filename, 'left_path'] = image_path
            else:
                F_image_df.loc[F_image_df['filename'] == filename, 'right_path'] = image_path

    else:
        print(f"Unknown image format: {image}")

len(F_image_df), len(L_image_df), len(R_image_df), sum([len(F_image_df), len(L_image_df), len(R_image_df)])

/tmp/ipykernel_3080039/2295192004.py:26: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '045_data/data/right_F0005_IND_F_42_0_01.png' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  F_image_df.loc[F_image_df['filename'] == filename, 'right_path'] = image_path


(1822, 1401, 1349, 4572)

In [488]:
F_image_df

,filename,left_path,right_path
0,F0005_IND_F_42_0_01.png,045_data/data/left_F0005_IND_F_42_0_01.png,045_data/data/right_F0005_IND_F_42_0_01.png
1,F0005_IND_F_42_0_02.png,045_data/data/left_F0005_IND_F_42_0_02.png,045_data/data/right_F0005_IND_F_42_0_02.png
2,F0002_IND_S_16_0_01.png,045_data/data/left_F0002_IND_S_16_0_01.png,045_data/data/right_F0002_IND_S_16_0_01.png
3,F0003_IND_GM_68_0_01.png,045_data/data/left_F0003_IND_GM_68_0_01.png,045_data/data/right_F0003_IND_GM_68_0_01.png
4,F0003_IND_M_39_0_02.png,045_data/data/left_F0003_IND_M_39_0_02.png,045_data/data/right_F0003_IND_M_39_0_02.png
...,...,...,...
1817,F0889_IND_F_45_0_02.png,045_data/data/left_F0889_IND_F_45_0_02.png,045_data/data/right_F0889_IND_F_45_0_02.png
1818,F0899_IND_M_47_0_02.png,045_data/data/left_F0899_IND_M_47_0_02.png,045_data/data/right_F0899_IND_M_47_0_02.png
1819,F0890_IND_M_46_0_02.png,045_data/data/left_F0890_IND_M_46_0_02.png,045_data/data/right_F0890_IND_M_46_0_02.png
1820,F0897_IND_M_50_0_02.png,045_data/data/left_F0897_IND_M_50_0_02.png,045_data/data/right_F0897_IND_M_50_0_02.png


In [489]:
left_merged = pd.merge(left_label_df, L_image_df, on='filename', how='outer')
left_merged.isna().sum()

filename      0
left_label    0
left_path     6
dtype: int64

In [490]:
right_merged = pd.merge(right_label_df, R_image_df, on='filename', how='outer')
right_merged.isna().sum()

filename        0
right_label     0
right_path     44
dtype: int64

In [491]:
F_merged = pd.merge(F_label_df, F_image_df, on='filename', how='outer')
F_merged.isna().sum()

filename       0
left_label     0
right_label    0
left_path      3
right_path     0
dtype: int64

In [492]:
F_merged = F_merged[['filename', 'left_path', 'right_path', 'left_label', 'right_label']]
F_merged

,filename,left_path,right_path,left_label,right_label
0,F0001_IND_D_18_0_01.png,045_data/data/left_F0001_IND_D_18_0_01.png,045_data/data/right_F0001_IND_D_18_0_01.png,0.0,0.0
1,F0001_IND_D_18_0_02.png,045_data/data/left_F0001_IND_D_18_0_02.png,045_data/data/right_F0001_IND_D_18_0_02.png,0.0,0.0
2,F0001_IND_GM_75_0_01.png,045_data/data/left_F0001_IND_GM_75_0_01.png,045_data/data/right_F0001_IND_GM_75_0_01.png,4.0,4.0
3,F0001_IND_GM_75_0_02.png,045_data/data/left_F0001_IND_GM_75_0_02.png,045_data/data/right_F0001_IND_GM_75_0_02.png,4.0,4.0
4,F0001_IND_M_45_0_01.png,045_data/data/left_F0001_IND_M_45_0_01.png,045_data/data/right_F0001_IND_M_45_0_01.png,1.0,1.0
...,...,...,...,...,...
1817,F0897_IND_M_50_0_02.png,045_data/data/left_F0897_IND_M_50_0_02.png,045_data/data/right_F0897_IND_M_50_0_02.png,2.0,2.0
1818,F0898_IND_D2_14_0_02.png,045_data/data/left_F0898_IND_D2_14_0_02.png,045_data/data/right_F0898_IND_D2_14_0_02.png,0.0,0.0
1819,F0898_IND_D_19_0_02.png,045_data/data/left_F0898_IND_D_19_0_02.png,045_data/data/right_F0898_IND_D_19_0_02.png,0.0,0.0
1820,F0898_IND_F_50_0_02.png,045_data/data/left_F0898_IND_F_50_0_02.png,045_data/data/right_F0898_IND_F_50_0_02.png,1.0,2.0


In [493]:
# LR_merged 만들자
left_merged['img_id'] = left_merged['filename'].apply(lambda x: x.replace('_-45_', '_LR_'))
right_merged['img_id'] = right_merged['filename'].apply(lambda x: x.replace('_45_', '_LR_'))
LR_merged = pd.merge(left_merged, right_merged, on='img_id', how='outer')
# LR_merged.rename(columns={'filename_x': 'left_filename', 'filename_y': 'right_filename'}, inplace=True)
LR_merged = LR_merged[['img_id', 'left_path', 'right_path', 'left_label', 'right_label']].rename(columns={'img_id': 'filename'})
LR_merged

,filename,left_path,right_path,left_label,right_label
0,F0001_IND_D_18_LR_01.png,045_data/data/left_F0001_IND_D_18_-45_01.png,NaN,0.0,0.0
1,F0001_IND_D_18_LR_02.png,045_data/data/left_F0001_IND_D_18_-45_02.png,NaN,0.0,0.0
2,F0001_IND_GM_75_LR_01.png,045_data/data/left_F0001_IND_GM_75_-45_01.png,NaN,4.0,4.0
3,F0001_IND_GM_75_LR_02.png,045_data/data/left_F0001_IND_GM_75_-45_02.png,NaN,4.0,4.0
4,F0001_IND_M_45_LR_01.png,045_data/data/left_F0001_IND_M_45_-45_01.png,NaN,1.0,NaN
...,...,...,...,...,...
1469,F0897_IND_M_50_LR_02.png,045_data/data/left_F0897_IND_M_50_-45_02.png,045_data/data/right_F0897_IND_M_50_45_02.png,2.0,2.0
1470,F0898_IND_D2_14_LR_02.png,045_data/data/left_F0898_IND_D2_14_-45_02.png,045_data/data/right_F0898_IND_D2_14_45_02.png,0.0,0.0
1471,F0898_IND_D_19_LR_02.png,045_data/data/left_F0898_IND_D_19_-45_02.png,045_data/data/right_F0898_IND_D_19_45_02.png,0.0,0.0
1472,F0898_IND_F_50_LR_02.png,045_data/data/left_F0898_IND_F_50_-45_02.png,045_data/data/right_F0898_IND_F_50_45_02.png,1.0,2.0


In [494]:
merged_df = pd.concat([F_merged, LR_merged], ignore_index=True)


In [495]:
import re

merged_df['user_id'] = merged_df['filename'].apply(lambda x:re.search(r'F[0-9]{4}_[A-Z]+_[A-Z]+[0-9]*', x).group(0))
merged_df

,filename,left_path,right_path,left_label,right_label,user_id
0,F0001_IND_D_18_0_01.png,045_data/data/left_F0001_IND_D_18_0_01.png,045_data/data/right_F0001_IND_D_18_0_01.png,0.0,0.0,F0001_IND_D
1,F0001_IND_D_18_0_02.png,045_data/data/left_F0001_IND_D_18_0_02.png,045_data/data/right_F0001_IND_D_18_0_02.png,0.0,0.0,F0001_IND_D
2,F0001_IND_GM_75_0_01.png,045_data/data/left_F0001_IND_GM_75_0_01.png,045_data/data/right_F0001_IND_GM_75_0_01.png,4.0,4.0,F0001_IND_GM
3,F0001_IND_GM_75_0_02.png,045_data/data/left_F0001_IND_GM_75_0_02.png,045_data/data/right_F0001_IND_GM_75_0_02.png,4.0,4.0,F0001_IND_GM
4,F0001_IND_M_45_0_01.png,045_data/data/left_F0001_IND_M_45_0_01.png,045_data/data/right_F0001_IND_M_45_0_01.png,1.0,1.0,F0001_IND_M
...,...,...,...,...,...,...
3291,F0897_IND_M_50_LR_02.png,045_data/data/left_F0897_IND_M_50_-45_02.png,045_data/data/right_F0897_IND_M_50_45_02.png,2.0,2.0,F0897_IND_M
3292,F0898_IND_D2_14_LR_02.png,045_data/data/left_F0898_IND_D2_14_-45_02.png,045_data/data/right_F0898_IND_D2_14_45_02.png,0.0,0.0,F0898_IND_D2
3293,F0898_IND_D_19_LR_02.png,045_data/data/left_F0898_IND_D_19_-45_02.png,045_data/data/right_F0898_IND_D_19_45_02.png,0.0,0.0,F0898_IND_D
3294,F0898_IND_F_50_LR_02.png,045_data/data/left_F0898_IND_F_50_-45_02.png,045_data/data/right_F0898_IND_F_50_45_02.png,1.0,2.0,F0898_IND_F


In [496]:
merged_df.isna().sum()

filename         0
left_path       76
right_path     125
left_label      67
right_label     81
user_id          0
dtype: int64

In [497]:
merged_df = merged_df[['user_id', 'filename', 'left_path', 'right_path', 'left_label', 'right_label']]
merged_df.head(3)

,user_id,filename,left_path,right_path,left_label,right_label
0,F0001_IND_D,F0001_IND_D_18_0_01.png,045_data/data/left_F0001_IND_D_18_0_01.png,045_data/data/right_F0001_IND_D_18_0_01.png,0.0,0.0
1,F0001_IND_D,F0001_IND_D_18_0_02.png,045_data/data/left_F0001_IND_D_18_0_02.png,045_data/data/right_F0001_IND_D_18_0_02.png,0.0,0.0
2,F0001_IND_GM,F0001_IND_GM_75_0_01.png,045_data/data/left_F0001_IND_GM_75_0_01.png,045_data/data/right_F0001_IND_GM_75_0_01.png,4.0,4.0


In [498]:
merged_df.isna().sum()

user_id          0
filename         0
left_path       76
right_path     125
left_label      67
right_label     81
dtype: int64

In [500]:
failed_paths = set()

for i, row in merged_df.iterrows():
    if str(row['left_path']).lower() != 'nan':
        if not os.path.exists(os.path.join(pig_root, row['left_path'])):
            print(f"Left path does not exist: {row['left_path']}")
            failed_paths.add(row['left_path'])
    else:
        failed_paths.add(row['left_path'])
    if str(row['right_path']).lower() != 'nan':
        if not os.path.exists(os.path.join(pig_root, row['right_path'])):
            print(f"Right path does not exist: {row['right_path']}")
            failed_paths.add(row['right_path'])
    else:
        failed_paths.add(row['right_path'])
len(failed_paths)

1

In [501]:
import json

json_path = os.path.join(pig_root, '045_data', 'label', 'pig_label.json')
os.makedirs(os.path.dirname(json_path), exist_ok=True)
with open(json_path, 'w') as f:
    json.dump(merged_df.to_dict(orient='records'), f, indent=2, ensure_ascii=False)

### 118_data (고화질 only)

In [502]:
import os
import pandas as pd
# meta data 가져오기

label_path = "/home/work/hocheol_dir/workspace/datasets/v0.2_data/pigment/118_data/NIA_age_face_pigmentation_250619.csv"
label_df = pd.read_csv(label_path)
label_df.head()

,filename,right_cheek,left_cheek,box_w,box_h
0,0001_1992_28_00000071_D.png,NaN,NaN,607.442444,721.450480
1,0002_1997_26_00000050_D.png,0.0,0.0,857.429579,957.556410
2,0002_1997_26_00000051_D.png,0.0,0.0,837.935589,900.689212
3,0003_1988_20_00000038_D.png,NaN,NaN,657.667152,728.690622
4,0003_1988_20_00000046_D.png,NaN,NaN,816.169738,902.846387


In [503]:
import math
label_df['valid'] = label_df.apply(lambda x: 0 if math.isnan(x['right_cheek']) or math.isnan(x['left_cheek']) else 1, axis=1)
label_df = label_df[label_df['valid'] == 1]

In [504]:
len(os.listdir(os.path.join(pig_root, '118_data/data')))

4963

In [505]:
label_df = label_df[['filename', 'right_cheek', 'left_cheek']]
label_df[['right_cheek', 'left_cheek']] = label_df[['right_cheek', 'left_cheek']].astype(int)
label_df

,filename,right_cheek,left_cheek
1,0002_1997_26_00000050_D.png,0,0
2,0002_1997_26_00000051_D.png,0,0
7,0003_1988_35_00000059_D.png,0,0
21,0010_1997_11_00000039_D.png,0,0
39,0013_1972_46_00000035_D.png,0,0
...,...,...,...
2455,0995_2003_20_00000039_D.png,0,0
2456,0995_2003_20_00000040_D.png,0,0
2457,0995_2003_20_00000042_D.png,0,0
2458,0995_2003_20_00000044_D.png,0,0


In [506]:
# image path 가져오기
import os
import re

pig_root = '/home/work/hocheol_dir/workspace/datasets/v0.2.2_data/pigment'

exts = set()

for image in os.listdir(os.path.join(pig_root, '118_data/data')):
    exts.add(image.split('.')[-1])
    
exts

{'png'}

In [507]:
id_set = set()
image_df = pd.DataFrame(columns=['filename', 'left_path', 'right_path'])

for image in os.listdir(os.path.join(pig_root, '118_data/data')):
    image_path = os.path.join('118_data/data', image)
    filename = image.replace('left_', '').replace('right_', '')
    if filename in label_df['filename'].values:
        if 'left' in image:
            if filename not in image_df['filename'].values:
                image_df.loc[len(image_df)] = (filename, image_path, None)
            else:
                image_df.loc[image_df['filename'] == filename, 'left_path'] = image_path
        elif 'right' in image:
            if filename not in image_df['filename'].values:
                image_df.loc[len(image_df)] = (filename, None, image_path)
            else:
                image_df.loc[image_df['filename'] == filename, 'right_path'] = image_path

len(image_df)

775

In [508]:
# image_df
label_df

,filename,right_cheek,left_cheek
1,0002_1997_26_00000050_D.png,0,0
2,0002_1997_26_00000051_D.png,0,0
7,0003_1988_35_00000059_D.png,0,0
21,0010_1997_11_00000039_D.png,0,0
39,0013_1972_46_00000035_D.png,0,0
...,...,...,...
2455,0995_2003_20_00000039_D.png,0,0
2456,0995_2003_20_00000040_D.png,0,0
2457,0995_2003_20_00000042_D.png,0,0
2458,0995_2003_20_00000044_D.png,0,0


In [509]:
merged_df = pd.merge(image_df, label_df, on='filename', how='inner')
len(merged_df)

775

In [510]:
failed_paths = set()
for idx, row in merged_df.iterrows():
    if not os.path.exists(os.path.join(pig_root, row['left_path'])):
        failed_paths.add(row['left_path'])
    if not os.path.exists(os.path.join(pig_root, row['right_path'])):
        failed_paths.add(row['right_path'])

len(failed_paths)

0

In [511]:
# 같은사람 연도 바뀌어도(다른 나이) 같은 user_id로 묶기
merged_df['user_id'] = merged_df['filename'].apply(lambda x:x.split('_')[0])
merged_df = merged_df[['user_id', 'filename', 'left_path', 'right_path', 'left_cheek', 'right_cheek']]
merged_df.sort_values(by=['user_id', 'filename'], ignore_index=True, inplace=True)

In [512]:
merged_df.rename(columns={'left_cheek': 'left_label', 'right_cheek': 'right_label'}, inplace=True)

In [513]:
import json

json_path = os.path.join(pig_root, '118_data', 'label', 'pig_label.json')
os.makedirs(os.path.dirname(json_path), exist_ok=True)
with open(json_path, 'w') as f:
    json.dump(merged_df.to_dict(orient='records'), f, indent=2, ensure_ascii=False)

### 999

In [514]:
import os
import pandas as pd
# meta data 가져오기

label_paths = [
    '/home/work/hocheol_dir/workspace/datasets/v0.2_data/pigment/999_data/NIA_online_pigment.csv',
    '/home/work/hocheol_dir/workspace/datasets/v0.2_data/pigment/999_data/NIA_SNUH_pigment.csv'
]
label_df = pd.concat([pd.read_csv(path) for path in label_paths], ignore_index=True)
label_df.head()

,filename,right_cheek,left_cheek
0,6101_4_F,0.0,0.0
1,6101_4_F_2,0.0,0.0
2,6101_4_L,NaN,0.0
3,6101_4_L_2,NaN,0.0
4,6101_4_R,0.0,NaN


In [515]:
len(label_df)

5826

In [516]:
len(os.listdir(os.path.join(pig_root, '999_data/data')))

7784

In [518]:
# image path 가져오기
import os
import re

pig_root = '/home/work/hocheol_dir/workspace/datasets/v0.2.2_data/pigment'

exts = set()

for image in os.listdir(os.path.join(pig_root, '999_data/data')):
    exts.add(image.split('.')[-1])
    
exts

{'png'}

In [519]:
label_df['filename'] = label_df['filename'].apply(lambda x: f"{os.path.splitext(x)[0]}.png")

In [520]:
label_df.sample(3)

,filename,right_cheek,left_cheek
5748,36020_03_F.png,3.0,3.0
3858,6266_4_F_2.png,1.0,0.0
467,6120_5_R_2.png,4.0,NaN


In [521]:
len(label_df)

5826

In [522]:
import os

F_img_df = pd.DataFrame(columns=['filename', 'left_path', 'right_path'])
L_img_df = pd.DataFrame(columns=['filename', 'left_path'])
R_img_df = pd.DataFrame(columns=['filename', 'right_path'])

for image in os.listdir(os.path.join(pig_root, '999_data/data')):
    image_path = os.path.join('999_data/data', image)
    filename = image.replace('left_', '').replace('right_', '')
    if '_F' in filename:
        if 'left' in image:
            if filename not in F_img_df['filename'].values:
                F_img_df.loc[len(F_img_df)] = (filename, image_path, None)
            else:
                F_img_df.loc[F_img_df['filename'] == filename, 'left_path'] = image_path
        elif 'right' in image:
            if filename not in F_img_df['filename'].values:
                F_img_df.loc[len(F_img_df)] = (filename, None, image_path)
            else:
                F_img_df.loc[F_img_df['filename'] == filename, 'right_path'] = image_path
    elif '_L' in filename:
        if 'left' in image:
            L_img_df.loc[len(L_img_df)] = (filename, image_path)
        else:
            print(f"Unknown image format: {image}")
    elif '_R' in filename:
        if 'right' in image:
            R_img_df.loc[len(R_img_df)] = (filename, image_path)
        else:
            print(f"Unknown image format: {image}")
    else:
        print(f"Unknown image format: {image}")

len(F_img_df), len(L_img_df), len(R_img_df)

(1996, 1879, 1914)

In [523]:
F_merged = pd.merge(F_img_df, label_df, on='filename', how='inner')
F_merged['user_id'] = F_merged['filename'].apply(lambda x: x.split('_')[0])
F_merged.rename(columns={'left_cheek': 'left_label', 'right_cheek': 'right_label'}, inplace=True)
F_merged = F_merged[['user_id', 'filename', 'left_path', 'right_path', 'left_label', 'right_label']]
F_merged

,user_id,filename,left_path,right_path,left_label,right_label
0,31001,31001_01_F.png,999_data/data/left_31001_01_F.png,999_data/data/right_31001_01_F.png,0.0,0.0
1,31001,31001_01_F_2.png,999_data/data/left_31001_01_F_2.png,999_data/data/right_31001_01_F_2.png,0.0,0.0
2,31002,31002_03_F_2.png,999_data/data/left_31002_03_F_2.png,999_data/data/right_31002_03_F_2.png,0.0,0.0
3,31002,31002_03_F.png,999_data/data/left_31002_03_F.png,999_data/data/right_31002_03_F.png,0.0,0.0
4,31003,31003_03_F.png,999_data/data/left_31003_03_F.png,999_data/data/right_31003_03_F.png,0.0,0.0
...,...,...,...,...,...,...
1991,6302,6302_7_F_2.png,999_data/data/left_6302_7_F_2.png,999_data/data/right_6302_7_F_2.png,0.0,0.0
1992,6302,6302_4_F.png,999_data/data/left_6302_4_F.png,999_data/data/right_6302_4_F.png,0.0,0.0
1993,6302,6302_7_F.png,999_data/data/left_6302_7_F.png,999_data/data/right_6302_7_F.png,0.0,0.0
1994,6302,6302_6_F.png,999_data/data/left_6302_6_F.png,999_data/data/right_6302_6_F.png,0.0,0.0


In [524]:
L_merged = pd.merge(L_img_df, label_df, on='filename', how='inner').drop(columns=['right_cheek'])

In [525]:
R_merged = pd.merge(R_img_df, label_df, on='filename', how='inner').drop(columns=['left_cheek'])

In [526]:
R_merged.head(2)

,filename,right_path,right_cheek
0,31011_03_R_2.png,999_data/data/right_31011_03_R_2.png,1.0
1,31013_03_R.png,999_data/data/right_31013_03_R.png,1.0


In [527]:
L_merged['id'] = L_merged['filename'].apply(lambda x:x.replace('_L', '_LR'))
R_merged['id'] = R_merged['filename'].apply(lambda x:x.replace('_R', '_LR'))
LR_merged = pd.merge(L_merged, R_merged, on='id', how='outer')
LR_merged.drop(columns=['filename_x', 'filename_y'], inplace=True)
LR_merged.rename(columns={'id':'filename'}, inplace=True)
LR_merged['user_id'] = LR_merged['filename'].apply(lambda x:x.split('_')[0])
LR_merged = LR_merged[['user_id', 'filename', 'left_path', 'right_path','left_cheek', 'right_cheek']]
LR_merged.rename(columns={'left_cheek': 'left_label', 'right_cheek': 'right_label'}, inplace=True)
LR_merged

,user_id,filename,left_path,right_path,left_label,right_label
0,31001,31001_01_LR.png,999_data/data/left_31001_01_L.png,999_data/data/right_31001_01_R.png,1.0,0.0
1,31001,31001_01_LR_2.png,999_data/data/left_31001_01_L_2.png,999_data/data/right_31001_01_R_2.png,1.0,0.0
2,31001,31001_02_LR.png,999_data/data/left_31001_02_L.png,999_data/data/right_31001_02_R.png,1.0,0.0
3,31001,31001_02_LR_2.png,NaN,999_data/data/right_31001_02_R_2.png,NaN,0.0
4,31001,31001_03_LR.png,999_data/data/left_31001_03_L.png,999_data/data/right_31001_03_R.png,1.0,0.0
...,...,...,...,...,...,...
1932,6302,6302_5_LR_2.png,999_data/data/left_6302_5_L_2.png,999_data/data/right_6302_5_R_2.png,0.0,0.0
1933,6302,6302_6_LR.png,999_data/data/left_6302_6_L.png,999_data/data/right_6302_6_R.png,0.0,0.0
1934,6302,6302_6_LR_2.png,999_data/data/left_6302_6_L_2.png,999_data/data/right_6302_6_R_2.png,0.0,0.0
1935,6302,6302_7_LR.png,999_data/data/left_6302_7_L.png,999_data/data/right_6302_7_R.png,0.0,0.0


In [532]:
LR_merged.sample(3)

,user_id,filename,left_path,right_path,left_label,right_label
1117,6197,6197_6_LR_2.png,999_data/data/left_6197_6_L_2.png,999_data/data/right_6197_6_R_2.png,0.0,0.0
1263,6215,6215_7_LR_2.png,999_data/data/left_6215_7_L_2.png,999_data/data/right_6215_7_R_2.png,0.0,1.0
1646,6266,6266_5_LR_2.png,999_data/data/left_6266_5_L_2.png,999_data/data/right_6266_5_R_2.png,0.0,1.0


In [533]:
F_merged
LR_merged

,user_id,filename,left_path,right_path,left_label,right_label
0,31001,31001_01_LR.png,999_data/data/left_31001_01_L.png,999_data/data/right_31001_01_R.png,1.0,0.0
1,31001,31001_01_LR_2.png,999_data/data/left_31001_01_L_2.png,999_data/data/right_31001_01_R_2.png,1.0,0.0
2,31001,31001_02_LR.png,999_data/data/left_31001_02_L.png,999_data/data/right_31001_02_R.png,1.0,0.0
3,31001,31001_02_LR_2.png,NaN,999_data/data/right_31001_02_R_2.png,NaN,0.0
4,31001,31001_03_LR.png,999_data/data/left_31001_03_L.png,999_data/data/right_31001_03_R.png,1.0,0.0
...,...,...,...,...,...,...
1932,6302,6302_5_LR_2.png,999_data/data/left_6302_5_L_2.png,999_data/data/right_6302_5_R_2.png,0.0,0.0
1933,6302,6302_6_LR.png,999_data/data/left_6302_6_L.png,999_data/data/right_6302_6_R.png,0.0,0.0
1934,6302,6302_6_LR_2.png,999_data/data/left_6302_6_L_2.png,999_data/data/right_6302_6_R_2.png,0.0,0.0
1935,6302,6302_7_LR.png,999_data/data/left_6302_7_L.png,999_data/data/right_6302_7_R.png,0.0,0.0


In [534]:
merged_df = pd.concat([F_merged, LR_merged], ignore_index=True)
merged_df

,user_id,filename,left_path,right_path,left_label,right_label
0,31001,31001_01_F.png,999_data/data/left_31001_01_F.png,999_data/data/right_31001_01_F.png,0.0,0.0
1,31001,31001_01_F_2.png,999_data/data/left_31001_01_F_2.png,999_data/data/right_31001_01_F_2.png,0.0,0.0
2,31002,31002_03_F_2.png,999_data/data/left_31002_03_F_2.png,999_data/data/right_31002_03_F_2.png,0.0,0.0
3,31002,31002_03_F.png,999_data/data/left_31002_03_F.png,999_data/data/right_31002_03_F.png,0.0,0.0
4,31003,31003_03_F.png,999_data/data/left_31003_03_F.png,999_data/data/right_31003_03_F.png,0.0,0.0
...,...,...,...,...,...,...
3928,6302,6302_5_LR_2.png,999_data/data/left_6302_5_L_2.png,999_data/data/right_6302_5_R_2.png,0.0,0.0
3929,6302,6302_6_LR.png,999_data/data/left_6302_6_L.png,999_data/data/right_6302_6_R.png,0.0,0.0
3930,6302,6302_6_LR_2.png,999_data/data/left_6302_6_L_2.png,999_data/data/right_6302_6_R_2.png,0.0,0.0
3931,6302,6302_7_LR.png,999_data/data/left_6302_7_L.png,999_data/data/right_6302_7_R.png,0.0,0.0


In [535]:
import json

json_path = os.path.join(pig_root, '999_data', 'label', 'pig_label.json')
os.makedirs(os.path.dirname(json_path), exist_ok=True)
with open(json_path, 'w') as f:
    json.dump(merged_df.to_dict(orient='records'), f, indent=2, ensure_ascii=False)